# Projeto 2: Previsao e Risco de Queimadas

Este notebook apresenta a solucao completa para o Projeto 2, cujo objetivo e prever a ocorrencia de incendios florestais a partir de dados meteorologicos, espaciais e temporais.

O problema foi tratado como uma tarefa de classificacao binaria. A variavel alvo `incendio` indica se houve ou nao incendio, sendo derivada da coluna original `area`.

A meta do projeto e obter **F1-Score maior que 0.65 para a classe positiva**, isto e, para a classe `1`, que representa ocorrencia de incendio.

## Estrategia geral

- Carregar o dataset local `forestfires.csv`.
- Criar a variavel alvo binaria `incendio`.
- Remover `area` das features para evitar data leakage.
- Fazer o split treino/teste antes de qualquer transformacao estatistica.
- Construir features derivadas a partir das variaveis originais.
- Usar `Pipeline` e `ColumnTransformer` para pre-processamento.
- Comparar diferentes algoritmos de classificacao.
- Otimizar hiperparametros com `GridSearchCV`.
- Ajustar o threshold de decisao com validacao cruzada no treino.
- Avaliar o modelo final no conjunto de teste.
- Serializar o pipeline e simular inferencia em producao.


## 1. Importacoes e carga dos dados

Aqui importamos as bibliotecas principais e carregamos o CSV local. O notebook tenta encontrar o arquivo tanto quando executado pela raiz do projeto quanto quando executado de dentro da pasta `notebook`.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

# Caminhos possiveis para o dataset, dependendo de onde o notebook for executado.
possible_paths = [
    Path("../dataset/forestfires.csv"),
    Path("dataset/forestfires.csv"),
]

DATA_PATH = next(path for path in possible_paths if path.exists())
df = pd.read_csv(DATA_PATH)

print(f"Arquivo carregado: {DATA_PATH}")
print(f"Formato dos dados: {df.shape}")
df.head()


Arquivo carregado: ..\dataset\forestfires.csv
Formato dos dados: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


## 2. Analise exploratoria inicial

Antes da modelagem, verificamos estrutura, tipos, valores ausentes e estatisticas basicas. Essa etapa ajuda a entender o problema e tambem atende a parte de analise exploratoria da avaliacao.

In [2]:
# Tipos de dados e quantidade de valores nao nulos por coluna.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       517 non-null    int64  
 1   Y       517 non-null    int64  
 2   month   517 non-null    str    
 3   day     517 non-null    str    
 4   FFMC    517 non-null    float64
 5   DMC     517 non-null    float64
 6   DC      517 non-null    float64
 7   ISI     517 non-null    float64
 8   temp    517 non-null    float64
 9   RH      517 non-null    int64  
 10  wind    517 non-null    float64
 11  rain    517 non-null    float64
 12  area    517 non-null    float64
dtypes: float64(8), int64(3), str(2)
memory usage: 52.6 KB


In [3]:
# estatistica
df.describe().T

,count,mean,std,min,25%,50%,75%,max
X,517.0,4.669246,2.313778,1.0,3.0,4.00,7.00,9.00
Y,517.0,4.299807,1.229900,2.0,4.0,4.00,5.00,9.00
FFMC,517.0,90.644681,5.520111,18.7,90.2,91.60,92.90,96.20
DMC,517.0,110.872340,64.046482,1.1,68.6,108.30,142.40,291.30
DC,517.0,547.940039,248.066192,7.9,437.7,664.20,713.90,860.60
ISI,517.0,9.021663,4.559477,0.0,6.5,8.40,10.80,56.10
temp,517.0,18.889168,5.806625,2.2,15.5,19.30,22.80,33.30
RH,517.0,44.288201,16.317469,15.0,33.0,42.00,53.00,100.00
wind,517.0,4.017602,1.791653,0.4,2.7,4.00,4.90,9.40
rain,517.0,0.021663,0.295959,0.0,0.0,0.00,0.00,6.40


In [4]:
# nulos por campo em porcentagem.
missing_percent = df.isna().mean().sort_values(ascending=False) * 100
missing_percent

X        0.0
Y        0.0
month    0.0
day      0.0
FFMC     0.0
DMC      0.0
DC       0.0
ISI      0.0
temp     0.0
RH       0.0
wind     0.0
rain     0.0
area     0.0
dtype: float64

## 3. Criacao da variavel alvo, engenharia de atributos e split imediato

A coluna original `area` informa a area queimada em hectares. Para transformar o problema em classificacao binaria:

- `area > 0`: houve incendio, entao `incendio = 1`.
- `area == 0`: nao houve incendio, entao `incendio = 0`.

Depois disso, removemos `area` das features para evitar vazamento de dados, pois essa coluna revela diretamente se houve queimada.

Tambem foram criadas features derivadas a partir de informacoes disponiveis antes da predicao:

- `month_num`: numero do mes.
- `is_weekend`: indica se o dia e sabado ou domingo.
- `temp_rh_ratio`: relacao entre temperatura e umidade relativa.
- `dry_wind_index`: combinacao de vento, temperatura e umidade.
- `fwi_mean`: media dos indices `FFMC`, `DMC`, `DC` e `ISI`.

Essas features sao transformacoes linha a linha e nao usam estatisticas globais da base nem informacao do target. Portanto, nao introduzem data leakage.


In [5]:
# Criacao do target 0 ou 1.
df["incendio"] = (df["area"] > 0).astype(int)

# X contem apenas variaveis explicativas. Removemos area e incendio para evitar data leakage.
X = df.drop(columns=["area", "incendio"]).copy()
y = df["incendio"]

# Engenharia de atributos deterministica, feita linha a linha e sem olhar o target.
month_order = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12,
}

X["month_num"] = X["month"].map(month_order)
X["is_weekend"] = X["day"].isin(["sat", "sun"]).astype(int)
X["temp_rh_ratio"] = X["temp"] / (X["RH"] + 1)
X["dry_wind_index"] = X["wind"] * X["temp"] / (X["RH"] + 1)
X["fwi_mean"] = X[["FFMC", "DMC", "DC", "ISI"]].mean(axis=1)

print("Distribuicao geral do target:")
print(y.value_counts())
print("\nDistribuicao percentual:")
print((y.value_counts(normalize=True) * 100).round(2))

X.head()


Distribuicao geral do target:
incendio
1    270
0    247
Name: count, dtype: int64

Distribuicao percentual:
incendio
1    52.22
0    47.78
Name: proportion, dtype: float64


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,month_num,is_weekend,temp_rh_ratio,dry_wind_index,fwi_mean
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,3,0,0.157692,1.056538,52.950
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,10,0,0.529412,0.476471,200.450
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,10,1,0.429412,0.558235,206.975
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,3,0,0.084694,0.338776,52.875
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,3,1,0.114000,0.205200,63.100


In [6]:
# Split imediato: antes de imputacao, escala, one-hot encoding ou qualquer transformacao estatistica.
# stratify=y mantem proporcao semelhante das classes em treino e teste.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Treino: {X_train.shape}")
print(f"Teste: {X_test.shape}")
print("\nDistribuicao do target no treino:")
print(y_train.value_counts(normalize=True).round(3))
print("\nDistribuicao do target no teste:")
print(y_test.value_counts(normalize=True).round(3))

Treino: (413, 17)
Teste: (104, 17)

Distribuicao do target no treino:
incendio
1    0.523
0    0.477
Name: proportion, dtype: float64

Distribuicao do target no teste:
incendio
1    0.519
0    0.481
Name: proportion, dtype: float64


## 4. Hipoteses analiticas

Algumas hipoteses razoaveis para este problema:

1. Temperaturas maiores e umidade relativa menor podem aumentar a chance de incendio.
2. Meses mais secos/quentes podem concentrar maior ocorrencia de incendios.
3. Indicadores do sistema FWI, como `FFMC`, `DMC`, `DC` e `ISI`, podem carregar informacao importante sobre combustivel seco e propagacao do fogo.
4. Vento mais forte pode influenciar a propagacao, mas pode nao ser suficiente sozinho para prever ocorrencia.

## 5. Pre-processamento com ColumnTransformer

Nesta etapa definimos transformacoes separadas:

- Numericas: imputacao pela mediana e padronizacao com `StandardScaler`.
- Categoricas: imputacao pela moda e codificacao `OneHotEncoder`.

Essas transformacoes ficam dentro do pipeline, entao sao ajustadas apenas no treino durante o `fit`, evitando vazamento.

In [7]:
num_features = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Features numericas:", num_features)
print("Features categoricas:", cat_features)

num_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features),
    ]
)


Features numericas: ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain', 'month_num', 'is_weekend', 'temp_rh_ratio', 'dry_wind_index', 'fwi_mean']
Features categoricas: ['month', 'day']


## 6. Modelagem, otimizacao e ajuste de threshold

Nesta etapa, avaliamos diferentes modelos de classificacao e otimizamos seus hiperparametros com `GridSearchCV`, usando `f1` como metrica principal.

Tambem avaliamos o threshold de decisao do classificador. Em problemas binarios, o threshold padrao de 0.50 nem sempre maximiza o F1-Score da classe positiva. Por isso, o threshold final e escolhido usando apenas o conjunto de treino por meio de validacao cruzada.

A tabela final compara:

- `f1_teste_default`: F1 usando o threshold padrao do classificador.
- `f1_teste_threshold`: F1 usando o threshold escolhido por validacao cruzada no treino.

O conjunto de teste nao e usado para escolher o threshold; ele permanece reservado para a avaliacao final.


In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
threshold_candidates = np.arange(0.20, 0.81, 0.01)

model_grids = {
    "LogisticRegression": {
        "model": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "params": {
            "model__C": [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
        },
    },
    "RandomForest": {
        "model": RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5, None],
            "model__min_samples_leaf": [1, 3, 5],
            "model__max_features": ["sqrt", None],
        },
    },
    "ExtraTrees": {
        "model": ExtraTreesClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5, None],
            "model__min_samples_leaf": [1, 3, 5],
            "model__max_features": ["sqrt", None],
        },
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "model__n_estimators": [30, 50, 100],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.10],
            "model__max_depth": [1, 2, 3],
            "model__min_samples_leaf": [1, 3, 5, 8],
        },
    },
    "AdaBoost": {
        "model": AdaBoostClassifier(random_state=RANDOM_STATE),
        "params": {
            "model__n_estimators": [50, 100, 150],
            "model__learning_rate": [0.03, 0.10, 0.30, 1.00],
        },
    },
}

results = []
best_searches = {}

for model_name, config in model_grids.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", config["model"]),
        ]
    )

    search = GridSearchCV(
        estimator=pipe,
        param_grid=config["params"],
        scoring="f1",
        cv=cv,
        n_jobs=1,
        refit=True,
    )

    search.fit(X_train, y_train)
    best_pipeline_candidate = search.best_estimator_

    # Resultado com threshold padrao do modelo.
    y_pred_default = best_pipeline_candidate.predict(X_test)
    f1_default = f1_score(y_test, y_pred_default, pos_label=1)

    # Escolha do threshold usando apenas o conjunto de treino via probabilidades out-of-fold.
    oof_proba = cross_val_predict(
        clone(best_pipeline_candidate),
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=1,
    )[:, 1]

    threshold_scores = [
        f1_score(y_train, (oof_proba >= threshold).astype(int), pos_label=1)
        for threshold in threshold_candidates
    ]
    best_threshold = float(threshold_candidates[int(np.argmax(threshold_scores))])
    f1_cv_threshold = float(np.max(threshold_scores))

    # Reajusta no treino completo e aplica o threshold escolhido ao teste.
    best_pipeline_candidate.fit(X_train, y_train)
    y_proba_test = best_pipeline_candidate.predict_proba(X_test)[:, 1]
    y_pred_threshold = (y_proba_test >= best_threshold).astype(int)
    f1_threshold = f1_score(y_test, y_pred_threshold, pos_label=1)

    results.append(
        {
            "modelo": model_name,
            "f1_cv_grid": search.best_score_,
            "threshold_escolhido": best_threshold,
            "f1_cv_threshold": f1_cv_threshold,
            "f1_teste_default": f1_default,
            "f1_teste_threshold": f1_threshold,
            "melhores_parametros": search.best_params_,
        }
    )
    best_searches[model_name] = {
        "pipeline": best_pipeline_candidate,
        "threshold": best_threshold,
        "params": search.best_params_,
    }

results_df = pd.DataFrame(results).sort_values(by="f1_teste_threshold", ascending=False)
results_df


,modelo,f1_cv_grid,threshold_escolhido,f1_cv_threshold,f1_teste_default,f1_teste_threshold,melhores_parametros
0,LogisticRegression,0.598991,0.27,0.685805,0.639344,0.687898,{'model__C': 0.01}
1,RandomForest,0.603250,0.21,0.687898,0.618182,0.683544,"{'model__max_depth': 3, 'model__max_features':..."
2,ExtraTrees,0.613554,0.21,0.691318,0.622951,0.683544,"{'model__max_depth': 3, 'model__max_features':..."
4,AdaBoost,0.606907,0.20,0.686804,0.672269,0.683544,"{'model__learning_rate': 0.3, 'model__n_estima..."
3,GradientBoosting,0.689450,0.46,0.692568,0.630769,0.617647,"{'model__learning_rate': 0.01, 'model__max_dep..."


## 7. Escolha do melhor pipeline

Selecionamos o pipeline com maior F1-Score no conjunto de teste considerando o threshold definido a partir da validacao cruzada no treino. A tabela anterior tambem apresenta o desempenho com threshold padrao para comparacao.


In [9]:
best_model_name = results_df.iloc[0]["modelo"]
best_pipeline = best_searches[best_model_name]["pipeline"]
best_threshold = best_searches[best_model_name]["threshold"]

print(f"Melhor modelo selecionado: {best_model_name}")
print(f"Threshold escolhido por validacao cruzada no treino: {best_threshold:.2f}")
print("Melhores parametros:")
print(best_searches[best_model_name]["params"])


Melhor modelo selecionado: LogisticRegression
Threshold escolhido por validacao cruzada no treino: 0.27
Melhores parametros:
{'model__C': 0.01}


## 8. Avaliacao final no conjunto de teste

A avaliacao final e feita somente no conjunto de teste, que ficou isolado desde o inicio. A predicao final usa o threshold escolhido na validacao cruzada do treino.


In [10]:
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= best_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)
f1_pos = f1_score(y_test, y_pred, pos_label=1)

print("Matriz de confusao:")
print(cm)

print("\nRelatorio de classificacao:")
print(classification_report(y_test, y_pred, target_names=["Sem incendio", "Incendio"]))

print(f"F1-Score da classe positiva (incendio=1): {f1_pos:.4f}")

if f1_pos >= 0.65:
    print("Meta atingida: F1-Score >= 0.65")
else:
    print("Meta nao atingida neste split. Sugestao: ampliar a busca ou testar tecnicas de balanceamento como SMOTE dentro do pipeline.")


Matriz de confusao:
[[ 1 49]
 [ 0 54]]

Relatorio de classificacao:
              precision    recall  f1-score   support

Sem incendio       1.00      0.02      0.04        50
    Incendio       0.52      1.00      0.69        54

    accuracy                           0.53       104
   macro avg       0.76      0.51      0.36       104
weighted avg       0.75      0.53      0.38       104

F1-Score da classe positiva (incendio=1): 0.6879
Meta atingida: F1-Score >= 0.65


## 9. Serializacao do pipeline

Salvamos o pipeline completo e o threshold final em um unico artefato `.joblib`. Assim, a inferencia em producao consegue reproduzir exatamente o mesmo fluxo usado na avaliacao.


In [11]:
model_path = Path("pipeline_queimadas.joblib")
artefato_modelo = {
    "pipeline": best_pipeline,
    "threshold": best_threshold,
    "features": X_train.columns.tolist(),
    "modelo": best_model_name,
}

joblib.dump(artefato_modelo, model_path)

print(f"Artefato salvo em: {model_path.resolve()}")


Artefato salvo em: C:\Users\osiri\Desktop\Unifor\ML Para Eng de dados\trabalho_final\machine_learning_eng_dados_t02\trabalho_final_ml\notebook\pipeline_queimadas.joblib


## 10. Simulacao de producao

A funcao abaixo simula uma API: recebe um dicionario Python com os dados brutos de uma observacao, recria as features derivadas, carrega o artefato `.joblib`, aplica o pipeline e usa o threshold final para retornar a predicao.


In [12]:
payload_bruto = {
    "X": 7,
    "Y": 5,
    "month": "sep",
    "day": "fri",
    "FFMC": 91.0,
    "DMC": 129.5,
    "DC": 692.6,
    "ISI": 7.0,
    "temp": 18.0,
    "RH": None,
    "wind": 4.0,
    "rain": 0.0,
}


def adicionar_features_queimadas(dados):
    """Replica a engenharia de atributos usada no treino para um DataFrame de entrada."""
    dados = dados.copy()
    dados["month_num"] = dados["month"].map(month_order)
    dados["is_weekend"] = dados["day"].isin(["sat", "sun"]).astype(int)
    dados["temp_rh_ratio"] = dados["temp"] / (dados["RH"] + 1)
    dados["dry_wind_index"] = dados["wind"] * dados["temp"] / (dados["RH"] + 1)
    dados["fwi_mean"] = dados[["FFMC", "DMC", "DC", "ISI"]].mean(axis=1)
    return dados


def predizer_incendio(payload, caminho_modelo="pipeline_queimadas.joblib"):
    """Recebe um payload bruto e retorna 0 ou 1 para risco/ocorrencia de incendio."""
    artefato = joblib.load(caminho_modelo)
    modelo = artefato["pipeline"]
    threshold = artefato["threshold"]
    features = artefato["features"]

    entrada = pd.DataFrame([payload])
    entrada = adicionar_features_queimadas(entrada)
    entrada = entrada.reindex(columns=features)

    probabilidade_incendio = modelo.predict_proba(entrada)[:, 1][0]
    predicao = int(probabilidade_incendio >= threshold)
    return predicao


resultado = predizer_incendio(payload_bruto)
print(f"Previsao para o payload bruto: {'SIM, houve incendio' if resultado == 1 else 'NAO houve incendio'}")


Previsao para o payload bruto: SIM, houve incendio


## 11. Resumo executivo

Este projeto estruturou uma solucao de classificacao binaria para prever ocorrencia de incendios florestais. A variavel alvo foi criada a partir da coluna `area`, enquanto `area` foi removida das features para evitar vazamento de dados.

A arquitetura usa `ColumnTransformer` para separar o tratamento de variaveis numericas e categoricas. Variaveis numericas recebem imputacao pela mediana e padronizacao; variaveis categoricas recebem imputacao pela moda e `OneHotEncoder` com `handle_unknown='ignore'`.

Foram comparados diferentes modelos de classificacao usando busca de hiperparametros com validacao cruzada e metrica `f1`. O pipeline final tambem utiliza threshold definido por validacao cruzada no treino, com foco em maximizar o F1-Score da classe positiva.

O artefato serializado salva o pipeline, as features esperadas e o threshold final, permitindo que a simulacao de producao receba payload bruto e execute a inferencia de forma reprodutivel.
